# Parameter-Efficient Diffusion with Neural Cellular Automata
#### _John Kalkhof, Arlene Kühn, Yannik Frisch, Anirban Mukhopadhyay_

### Imports

In [ ]:
import torch
from src.datasets.png_Dataset import png_Dataset
from src.models.Model_FourierDiffNCA import FourierDiffNCA
from src.utils.Experiment import Experiment 
from src.agents.Agent_Diffusion import Agent_Diffusion

### Model Configuration

In [ ]:
config = [{
    'img_path': r"data_path", 
    'name': r'FourierDiffNCA_Run1', 
    'device':"cuda:0",
    'unlock_CPU': True,
    # Optimizer
    'lr': 16e-4,
    'lr_gamma': 0.9999,
    'betas': (0.9, 0.99),
    # Training
    'save_interval': 1,
    'evaluate_interval': 1,
    'n_epoch': 30,
    'batch_size': 16,
    # Model
    'channel_n': 96,       
    'batch_duplication': 1,
    'inference_steps': 10,
    'cell_fire_rate': 0.1,
    'input_channels': 3,
    'output_channels': 3,
    'hidden_size':  512,
    'schedule': 'cosine',
    # Data
    'input_size': (64, 64),
    'data_split': [0.8, 0.1, 0.1],
    'timesteps': 300,
    '2D': True,
    'unlock_CPU': True,
}
]
config[0]['label_path'] = config[0]['img_path']

### Model and Datasetup Setup

In [ ]:
dataset = png_Dataset(buffer=True)
device = torch.device(config[0]['device'])

ca0 = FourierDiffNCA(config[0]['channel_n'], config[0]['cell_fire_rate'], device, hidden_size=config[0]['hidden_size'], input_channels=config[0]['input_channels'], img_size=config[0]['input_size'][0],).to(device)
ca = [ca0]

agent = Agent_Diffusion(ca)
exp = Experiment(config, dataset, ca, agent)
dataset.set_experiment(exp)
exp.set_model_state('train')
data_loader = torch.utils.data.DataLoader(dataset, shuffle=True, batch_size=exp.get_from_config('batch_size'))

print("PARAMETERS", sum(p.numel() for p in ca0.parameters() if p.requires_grad))
 

### Start Training

In [ ]:
agent.train(data_loader)

### Generate Samples After Training

In [ ]:
agent.generateSamples(samples=1)